In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from Log_Extractor import LogExtractor


# 한글 폰트 설정
plt.rc('font', family='AppleGothic') 
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import os

# ==========================================
# ⚙️ 1. 설정 및 매개변수 (여기만 바꾸면 14개 펌프 무한 확장 가능!)
# ==========================================
PUMP_ID = 'P1' # 나중에 P2, P3 등으로 변경
TANK_ID = 'P1' # 나중에 T2, T3 등으로 변경
ROBOT_ID = 'RB1' # 나중에 R2, R3 등으로 변경
MODEL_SAVE_DIR = './models' # 모델과 스케일러를 저장할 폴더
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# ==========================================
# 📅 Train / Valid / Test 기간 분리 (겹치지 않음!)
# 전체 가용 구간: 2026-09-10 07:00 ~ 2026-09-18 11:00 (8일 4시간)
# ==========================================
TRAIN_START, TRAIN_END = "2026-09-10T07:00:00Z", "2026-09-16T07:00:00Z"  # 6일 - 학습
VALID_START, VALID_END = "2026-09-16T07:00:00Z", "2026-09-17T07:00:00Z"  # 1일 - 검증
TEST_START,  TEST_END  = "2026-09-17T07:00:00Z", "2026-09-18T11:00:00Z"  # 1일 4시간 - 테스트(이상탐지)

In [ ]:
# 한글 폰트 설정
plt.rc('font', family='Malgun Gothic')  # Windows 기본 한글 폰트
plt.rcParams['axes.unicode_minus'] = False

# 1. 추출기 가동 (어제 하루치 데이터만 가져오기)
# extractor는 이미 위에서 선언했다고 가정합니다.
target_tags = [
    f'Pump_BuildUp_{PUMP_ID}' , f'{ROBOT_ID}_Robot_Num', 
    f'g_s_SV_{PUMP_ID}', f'Ana_Out_{PUMP_ID}', 
    f'Scale_Out___PT_{PUMP_ID}', f'Scale_Out___FT_{PUMP_ID}', 
    f'TK_Temp_PV_{TANK_ID}', f'TK_Level_PV_{TANK_ID}'
]

In [ ]:
# 1. 추출기 가동 (어제 하루치 데이터만 가져오기)
# extractor는 이미 위에서 선언했다고 가정합니다.
target_tags += [
    'Emergency_Sw', '전진_Actual_Wagon_Num', 'g_s_SV_P1', 
    'Ana_Out_P1', 'Scale_Out___PT_P1', 'Scale_Out___FT_P1', 'TK_Temp_PV_P1', 'TK_Level_PV_P1'
]

In [ ]:

extractor = LogExtractor()
# -24h 로 하루치만 가볍게 뽑습니다.
All_df = extractor.get_data(start_time="2026-09-10T00:00:00Z", end_time="2026-09-22T00:00:00Z", target_tags=None)

# 결측치 방어를 위해 앞선 값으로 채우기
df = All_df.ffill().fillna(0).copy()